In [1]:
import numpy as np
import pandas as pd
import json

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import roc_auc_score

In [2]:
def temporal_split_with_gap(df, min_date, max_date, gap_size=0, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    """
    Splits a DataFrame into three buckets based on the temporal order of the 'Date' column.
    Also is adding 

    Args:
        df (DataFrame): The DataFrame to split.
        min_date (str or Timestamp): Minimum date in the DataFrame.
        max_date (str or Timestamp): Maximum date in the DataFrame.
        gap_size (int) - how many days gap betweeen sets parts
        train_prop (float): Proportion of data for training set (default: 0.6).
        val_prop (float): Proportion of data for validation set (default: 0.2).
        test_prop (float): Proportion of data for test set (default: 0.2).

    Returns:
        DataFrame: The input DataFrame with a new column 'split' indicating the split for each row.
    """
    # Define the date intervals
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)

    mask = (df['date'] <= train_end - pd.Timedelta(days=gap_size))
    df.loc[mask, 'split'] = 'train'
    mask = ((df['date'] > train_end) & (df['date'] <= val_end - pd.Timedelta(days=gap_size)))
    df.loc[mask, 'split'] = 'validation'
    mask = (df['date'] > val_end)
    df.loc[mask, 'split'] = 'test'

    return df

**Preparing data**

Downloading all features

In [3]:
df_features = pd.read_parquet('../data/features.parquet')

with open('../data/feature_set.json', encoding='utf-8') as f:
    feature_sets = json.load(f)
TO_PREDICT = feature_sets['TO_PREDICT']
NUMERICAL = feature_sets['NUMERICAL']
DUMMIES = feature_sets['DUMMIES']

Splitting datasets

In [4]:
df_full = df_features.copy()
min_date_df = df_full.date.min()
max_date_df = df_full.date.max()

df_full = temporal_split_with_gap(df_full,
                                 min_date = min_date_df,
                                 max_date = max_date_df,
                                 gap_size = 45)
len(df_full[df_full['split'] == 'train'])

59917

In [5]:
#preparing X sets
X_train = df_full[NUMERICAL + DUMMIES][df_full.split == 'train'].copy()
X_val = df_full[NUMERICAL + DUMMIES][df_full.split == 'validation'].copy()
X_test = df_full[NUMERICAL + DUMMIES][df_full.split == 'test'].copy()

#preparing y sets
y_train = df_full['is_positive_growth_30d_future'][df_full.split == 'train'].copy()
y_val = df_full['is_positive_growth_30d_future'][df_full.split == 'validation'].copy()
y_test = df_full['is_positive_growth_30d_future'][df_full.split == 'test'].copy()

#we don't need split anymore
df_full = df_full.drop(columns=['split'])

In [6]:
print(f'Base rate train set {round(y_train.mean(),2)}')
print(f'Base rate validation set {round(y_val.mean(),2)}')
print(f'Base rate test set {round(y_test.mean(),2)}')

Base rate train set 0.57
Base rate validation set 0.58
Base rate test set 0.63


**Models**

*Decision tree*

In [7]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred = dt.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, y_pred)

0.5113569226143864

In [13]:
min_samples_leaf_set = [200]
for max_depth in range(2, 16):
    for min_samples_leaf in min_samples_leaf_set:
        dt = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        dt.fit(X_train, y_train)
        y_pred = dt.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_pred)
        y_pred_class = (y_pred >= 0.8).astype(int)
        
        precision = precision_score(y_val, y_pred_class)
        print(f'For max_depth={max_depth} mean={y_pred_class.mean()} roc_auc_score={roc_auc} and precision_score={precision}')


For max_depth=2 mean=0.9247311827956989 roc_auc_score=0.510649654953015 and precision_score=0.5894996476391825
For max_depth=3 mean=0.9247311827956989 roc_auc_score=0.5113260602241582 and precision_score=0.5894996476391825
For max_depth=4 mean=0.7608341479309221 roc_auc_score=0.5442020985114386 and precision_score=0.6122055674518201
For max_depth=5 mean=0.5964483545128706 roc_auc_score=0.525252645535015 and precision_score=0.5919147773832286
For max_depth=6 mean=0.5964483545128706 roc_auc_score=0.5164446832349815 and precision_score=0.5919147773832286
For max_depth=7 mean=0.5964483545128706 roc_auc_score=0.5137235490197449 and precision_score=0.5919147773832286
For max_depth=8 mean=0.5964483545128706 roc_auc_score=0.5135095060531799 and precision_score=0.5919147773832286
For max_depth=9 mean=0.5964483545128706 roc_auc_score=0.5135095060531799 and precision_score=0.5919147773832286
For max_depth=10 mean=0.5964483545128706 roc_auc_score=0.5135095060531799 and precision_score=0.5919147773

Best depth is 4: AUC 0.544, precision 0.612 vs validation base rate 0.58. Deeper trees overfit (AUC goes down), from depth 8 tree stops growing because of min_samples_leaf=200. Improvement is small, but it's typical for stocks. 

*Random forest*

In [21]:
min_samples_leaf_set = [200]
n_estimators_set = [50, 100, 200, 500, 1000, 1500, 2000]

for n in n_estimators_set:
    for min_samples_leaf in min_samples_leaf_set:
        dt = RandomForestClassifier(n_estimators=n,
                                    min_samples_leaf=min_samples_leaf,
                                    n_jobs=-1,
                                    random_state=42)
        dt.fit(X_train, y_train)
        y_pred = dt.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_pred)
        y_pred_class = (y_pred >= 0.5).astype(int)
        
        precision = precision_score(y_val, y_pred_class)
        print(f'For n_estimators={n} mean={y_pred_class.mean()} roc_auc_score={roc_auc} and precision_score={precision}')

For n_estimators=50 mean=1.0 roc_auc_score=0.5550153241326631 and precision_score=0.5839035516454871
For n_estimators=100 mean=1.0 roc_auc_score=0.5786962546495693 and precision_score=0.5839035516454871
For n_estimators=200 mean=1.0 roc_auc_score=0.5811351482093914 and precision_score=0.5839035516454871
For n_estimators=500 mean=1.0 roc_auc_score=0.6012342115658911 and precision_score=0.5839035516454871
For n_estimators=1000 mean=1.0 roc_auc_score=0.6044711300638355 and precision_score=0.5839035516454871
For n_estimators=1500 mean=1.0 roc_auc_score=0.6053858312241303 and precision_score=0.5839035516454871
For n_estimators=2000 mean=1.0 roc_auc_score=0.6075404084279561 and precision_score=0.5839035516454871


We can see that AUC growth from 50 to 500 trees, after 500 it is plateau so we will leave only from 100 to 1000 for testing max_depth

In [22]:
min_samples_leaf_set = [200]
n_estimators_set = [100, 200, 500, 1000]
results = []

for n in n_estimators_set:
    for min_samples_leaf in min_samples_leaf_set:
        for max_depth in range(2, 16):
            dt = RandomForestClassifier(n_estimators=n,
                                        max_depth=max_depth,        
                                        min_samples_leaf=min_samples_leaf,
                                        n_jobs=-1,
                                        random_state=42)
            dt.fit(X_train, y_train)
            y_pred = dt.predict_proba(X_val)[:, 1]
            roc_auc = roc_auc_score(y_val, y_pred)
            y_pred_class = (y_pred >= 0.5).astype(int)
        
            precision = precision_score(y_val, y_pred_class)
            results.append({'n_estimators': n, 'min_samples_leaf': min_samples_leaf,
                'max_depth': max_depth, 'roc_auc': roc_auc,
                'precision': precision, 'signals_share': y_pred_class.mean()})

df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)

In [25]:
dt = RandomForestClassifier(n_estimators=100,
                            max_depth=3,        
                            min_samples_leaf=200,
                            n_jobs=-1,
                            random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_pred)
y_pred_class = (y_pred >= 0.5).astype(int)
        
precision = precision_score(y_val, y_pred_class)
pd.Series(y_pred).describe()

count    12276.000000
mean         0.628029
std          0.020421
min          0.570184
25%          0.617889
50%          0.627973
75%          0.637033
max          0.678258
dtype: float64

In [26]:
proba_set = [0.62, 0.63, 0.64, 0.65, 0.66]
for proba in proba_set:
    y_pred_class = (y_pred >= proba).astype(int)
    precision = precision_score(y_val, y_pred_class)
    print(f'proba={proba} precision={precision}')

proba=0.62 precision=0.635552039778506
proba=0.63 precision=0.6786438179473006
proba=0.64 precision=0.8060721062618595
proba=0.65 precision=0.8253706754530478
proba=0.66 precision=0.846830985915493


In [27]:
importances = pd.Series(dt.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(15)


gr_10y            0.147223
de_10y            0.100634
deposit_rate      0.097395
spread_gr_de      0.086363
month_8           0.056282
month_12          0.041819
month_3           0.036439
month_11          0.035779
daily_turnover    0.032089
month_9           0.025847
month_2           0.023396
trix              0.023326
roc               0.018623
natr              0.017749
growth_252d       0.016934
dtype: float64

In [28]:
#features without macro and month
macro_columns = ['gr_10y', 'de_10y', 'spread_gr_de', 'deposit_rate']
features_b = [col for col in X_train.columns if col not in macro_columns and not col.startswith('month_')]


#the same forest as the best one
dt = RandomForestClassifier(n_estimators=100,
                            max_depth=3,
                            min_samples_leaf=200,
                            n_jobs=-1,
                            random_state=42)
dt.fit(X_train[features_b], y_train)
y_pred = dt.predict_proba(X_val[features_b])[:, 1]
roc_auc = roc_auc_score(y_val, y_pred)
print(f'roc_auc_score without macro and month={roc_auc}')


#thresholds by quantiles: top 50%, 25%, 10%
quantile_set = [0.5, 0.75, 0.9]


for q in quantile_set:
    proba = pd.Series(y_pred).quantile(q)
    y_pred_class = (y_pred >= proba).astype(int)
    precision = precision_score(y_val, y_pred_class)
    print(f'For quantile={q} proba={proba} mean={y_pred_class.mean()} precision_score={precision}')


roc_auc_score without macro and month=0.5371762753759859
For quantile=0.5 proba=0.5633557931961363 mean=0.5004072987943956 precision_score=0.5993814097346574
For quantile=0.75 proba=0.5674104941019773 mean=0.25008145975887913 precision_score=0.6299674267100978
For quantile=0.9 proba=0.5738755920316222 mean=0.10019550342130987 precision_score=0.6560975609756098


Model A (all features): AUC 0.638, most important are macro and month, so it predicts good periods for the whole market, not good stocks. Model B (without macro and month): AUC 0.537, precision grows with threshold: 0.60 → 0.63 → 0.66 vs base rate 0.584. Both go to test, and both can be used as two strategies in simulation. 
